In [ ]:
# optional but recommended
import shutil
shutil.rmtree("faiss_index", ignore_errors=True)

In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters faiss-cpu pypdf transformers

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_name = list(uploaded.keys())[0]

loader = PyPDFLoader(file_name)
documents = loader.load()

print("Pages loaded:", len(documents))

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings()

vectorstore = FAISS.from_documents(chunks, embeddings)

print("Vector DB created")

In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
!pip install -U transformers

In [ ]:
from transformers import pipeline

pipe = pipeline(
    task="text-generation",
    model="google/flan-t5-base",
    max_new_tokens=100
)

print("Model loaded successfully!")

In [ ]:
def ask_pdf(query):

    docs = retriever.invoke(query)

    context = "\n".join([doc.page_content for doc in docs[:3]])

    prompt = f"""
Answer the question based only on the context below.

Context:
{context}

Question:
{query}
"""

    result = pipe(prompt)

    return result[0]["generated_text"]

In [ ]:
print(ask_pdf("What is this document about?"))

In [ ]:
print(ask_pdf("Summarize this document"))